# Blindspot · Training, Evaluation, and Deployment

This Colab notebook is the end-to-end training path for Blindspot:

1. audit the real dataset and held-out split
2. run real-data baseline calibration
3. generate a small oracle SFT warm-start on the 13 training users
4. run GRPO on top of the SFT adapter
5. evaluate on held-out users
6. precompute both demo caches and deploy the CPU-only Hugging Face Space

The intended hardware target is a single A100 Colab runtime.

## 1. Setup

In [ ]:
%%bash
pip install -q --upgrade unsloth trl 'openenv-core[core]' vllm matplotlib seaborn pandas datasets peft accelerate bitsandbytes requests
git clone https://github.com/vasarlalikhilavinash/blindspot-env || (cd blindspot-env && git pull)
cd blindspot-env
mkdir -p plots training/checkpoints
(nohup uvicorn server.app:app --host 0.0.0.0 --port 8000 --log-level warning > /tmp/blindspot.log 2>&1 &) || true
sleep 6
curl -s http://localhost:8000/state | head -c 300 || true

In [ ]:
import json
from pathlib import Path
import pandas as pd

DATA = Path('blindspot-env/data')
splits = json.load(open(DATA / 'user_splits.json'))
users = json.load(open(DATA / 'user_summaries.json'))
catalog = json.load(open(DATA / 'concept_catalog.json'))
paths = json.load(open(DATA / 'reading_paths.json'))
adoption = json.load(open(DATA / 'ground_truth_adoption.json'))
comp = json.load(open(DATA / 'comprehension_scores.json'))

stats = {
    'users': len(users),
    'train_users': len(splits['train']),
    'test_users': len(splits['test']),
    'concepts': len(catalog),
    'reading_paths': len(paths),
    'adoption_pairs': sum(len(v) for v in adoption.values()),
    'comprehension_pairs': sum(len(v) for v in comp.values()),
}

display(pd.DataFrame(list(stats.items()), columns=['artifact', 'value']))
print('train users:', splits['train'])
print('test users :', splits['test'])

## 2. Real-data calibration

In [ ]:
%%bash
cd blindspot-env
python scripts/baseline_eval.py

In [ ]:
import json
import pandas as pd

cal = json.load(open('blindspot-env/data/baseline_calibration.json'))
rows = []
for name, vals in cal.items():
    if name.startswith('_'):
        continue
    rows.append({
        'policy': name,
        'mean_reward': round(vals['mean'], 3),
        'std': round(vals['std'], 3),
        'n': vals['n'],
    })
display(pd.DataFrame(rows))

## 3. SFT warm-start

Before GRPO, build a small oracle demonstration set from the 13 training users.
This gives the model a clean action prior instead of starting from raw JSON emission.

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, 'blindspot-env')

from models import BlindspotAction
from server.blindspot_environment import BlindspotEnvironment
from training.generate_sft_traces import SYSTEM_PROMPT, render_obs

env = BlindspotEnvironment()
data = env._data
splits = json.load(open('blindspot-env/data/user_splits.json'))
train_users = splits['train']
out_path = Path('blindspot-env/training/sft_traces.jsonl')


def _lookup(mapping, key, default=0.0):
    if key in mapping:
        return mapping[key]
    skey = str(key)
    if skey in mapping:
        return mapping[skey]
    return default


def score_concept(uid, cid):
    adoption = data.adoption.get(uid, {})
    comprehension = data.comprehension.get(uid, {})
    novelty = data.novelty
    a = float(_lookup(adoption, cid, 0.0))
    if a < 1e-6:
        return -0.1
    bonus = 0.5 * (1.0 if bool(_lookup(novelty, cid, False)) else 0.0)
    onboarding = float(_lookup(comprehension, cid, 0.0))
    return a + bonus + onboarding


n_traces = 0
with out_path.open('w', encoding='utf-8') as handle:
    for uid in train_users:
        for seed in range(4):
            obs = env.reset(seed=seed, user_id=uid)
            ranked = sorted(
                [c.concept_id for c in obs.candidate_concepts],
                key=lambda cid: score_concept(uid, cid),
                reverse=True,
            )
            inspect_targets = ranked[:3]
            surface_targets = ranked[:3]

            messages = [
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': render_obs(obs)},
            ]

            for cid in inspect_targets:
                action = {'type': 'inspect', 'concept_id': cid}
                messages.append({'role': 'assistant', 'content': json.dumps(action)})
                obs = env.step(BlindspotAction(**action))
                if obs.done:
                    break
                messages.append({'role': 'user', 'content': render_obs(obs)})

            if not obs.done:
                for cid in surface_targets:
                    action = {'type': 'surface', 'concept_id': cid}
                    messages.append({'role': 'assistant', 'content': json.dumps(action)})
                    obs = env.step(BlindspotAction(**action))
                    if obs.done:
                        break
                    messages.append({'role': 'user', 'content': render_obs(obs)})

            if not obs.done:
                action = {'type': 'stop'}
                messages.append({'role': 'assistant', 'content': json.dumps(action)})
                obs = env.step(BlindspotAction(**action))

            handle.write(json.dumps({
                'user_id': uid,
                'seed': seed,
                'messages': messages,
                'final_reward': float(obs.reward_breakdown.total if obs.reward_breakdown else obs.reward),
            }) + '\n')
            n_traces += 1

print(f'wrote {n_traces} oracle traces to {out_path}')

In [ ]:
%%bash
cd blindspot-env
python training/sft_train.py --backend transformers --base-model unsloth/Qwen3.5-9B-bnb-4bit --epochs 1 --batch-size 1

## 4. GRPO training

In [ ]:
from unsloth import FastLanguageModel
import os
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen3.5-9B-bnb-4bit',
    max_seq_length=4096 + 128,
    load_in_4bit=True,
)

if os.path.isdir('blindspot-env/training/checkpoints/sft'):
    model.load_adapter('blindspot-env/training/checkpoints/sft')
    print('✓ attached SFT warm-start adapter')

FastLanguageModel.for_training(model)

if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f'GPU: {torch.cuda.get_device_name(0)} | free {free/1e9:.1f}/{total/1e9:.1f} GB')

In [ ]:
import json
import random
import requests
from datasets import Dataset
from training.grpo_train import SYSTEM_PROMPT, render_obs, parse_action

ENV_URL = 'http://localhost:8000'
split = json.load(open('blindspot-env/data/user_splits.json'))
train_users = split['train']
test_users = split['test']

rng = random.Random(0)
rows = []
for _ in range(256):
    uid = rng.choice(train_users)
    seed = rng.randrange(1_000_000)
    obs = requests.post(f'{ENV_URL}/reset', json={'user_id': uid, 'seed': seed}).json()
    obs = obs.get('observation', obs)
    rows.append({
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': render_obs(obs)},
        ],
        'user_id': uid,
        'seed': seed,
    })

ds = Dataset.from_list(rows)
print(f'built {len(ds)} GRPO prompts across {len(set(r["user_id"] for r in rows))} training users')

In [ ]:
import requests

REWARD_LOG = []


def reward_fn(prompts, completions, user_id=None, seed=None, **kwargs):
    rewards = []
    uids = user_id if isinstance(user_id, list) else [user_id] * len(completions)
    seeds = seed if isinstance(seed, list) else [seed] * len(completions)

    for prompt_msgs, completion, uid, sd in zip(prompts, completions, uids, seeds):
        text = completion if isinstance(completion, str) else completion[-1].get('content', '')
        action = parse_action(text)
        if not action or 'type' not in action:
            rewards.append(-0.05)
            REWARD_LOG.append((-0.05, {}))
            continue

        try:
            payload = {'user_id': uid, 'seed': sd}
            requests.post(f'{ENV_URL}/reset', json=payload).raise_for_status()
            step_resp = requests.post(f'{ENV_URL}/step', json={'action': action}).json()
            stop_resp = requests.post(f'{ENV_URL}/step', json={'action': {'type': 'stop'}}).json()
            br = (stop_resp.get('observation', stop_resp) or {}).get('reward_breakdown') or {}
            total = float(br.get('total', step_resp.get('reward', 0.0) or 0.0))
            rewards.append(total)
            REWARD_LOG.append((total, br))
        except Exception:
            rewards.append(-0.1)
            REWARD_LOG.append((-0.1, {}))

    return rewards

In [ ]:
from trl import GRPOConfig, GRPOTrainer

cfg = GRPOConfig(
    output_dir='blindspot-env/training/checkpoints/grpo',
    learning_rate=5e-6,
    max_steps=400,
    num_generations=8,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    max_prompt_length=4096,
    max_completion_length=64,
    logging_steps=5,
    save_steps=100,
    bf16=True,
    report_to='none',
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_fn],
    args=cfg,
    train_dataset=ds,
)

trainer.train()
trainer.save_model('blindspot-env/training/checkpoints/grpo')
print('✓ GRPO training complete')

## 5. Held-out evaluation

In [ ]:
import pandas as pd
import requests
import torch

FastLanguageModel.for_inference(model)


def trained_episode(user_id, seed):
    obs = requests.post(f'{ENV_URL}/reset', json={'user_id': user_id, 'seed': seed}).json()
    obs = obs.get('observation', obs)
    msgs = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': render_obs(obs)},
    ]
    inputs = tokenizer.apply_chat_template(msgs, return_tensors='pt', add_generation_prompt=True).to(model.device)
    with torch.inference_mode():
        out = model.generate(inputs, max_new_tokens=64, do_sample=False, temperature=0.0)
    completion = tokenizer.decode(out[0, inputs.shape[1]:], skip_special_tokens=True)
    action = parse_action(completion) or {'type': 'stop'}

    requests.post(f'{ENV_URL}/reset', json={'user_id': user_id, 'seed': seed}).raise_for_status()
    requests.post(f'{ENV_URL}/step', json={'action': action}).raise_for_status()
    rs = requests.post(f'{ENV_URL}/step', json={'action': {'type': 'stop'}}).json()
    br = (rs.get('observation', rs) or {}).get('reward_breakdown') or {}
    return br, action, completion


results = []
all_users = train_users + test_users
for uid in all_users:
    split_name = 'test' if uid in test_users else 'train'
    for seed in range(5):
        br, action, completion = trained_episode(uid, seed)
        results.append({
            'split': split_name,
            'user': uid,
            'seed': seed,
            'action_type': action.get('type'),
            'concept_id': action.get('concept_id'),
            'completion': completion,
            **br,
        })

df_eval = pd.DataFrame(results)
summary = df_eval.groupby('split')['total'].agg(['mean', 'std', 'count'])
summary.loc['all'] = {
    'mean': df_eval['total'].mean(),
    'std': df_eval['total'].std(),
    'count': len(df_eval),
}
display(summary)

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd

cal = json.load(open('blindspot-env/data/baseline_calibration.json'))
rows = []
for name, vals in cal.items():
    if name.startswith('_'):
        continue
    rows.append({'policy': name, 'mean': vals['mean'], 'std': vals['std']})

rows.append({
    'policy': 'GRPO (train users)',
    'mean': float(df_eval[df_eval['split'] == 'train']['total'].mean()),
    'std': float(df_eval[df_eval['split'] == 'train']['total'].std()),
})
rows.append({
    'policy': 'GRPO (held-out test)',
    'mean': float(df_eval[df_eval['split'] == 'test']['total'].mean()),
    'std': float(df_eval[df_eval['split'] == 'test']['total'].std()),
})

score_df = pd.DataFrame(rows)
display(score_df)

fig, ax = plt.subplots(figsize=(10, 4.8))
ax.bar(score_df['policy'], score_df['mean'], yerr=score_df['std'], capsize=5,
       color=['#888888', '#cc7733', '#3377cc', '#aa4488', '#22aa66', '#22aa66', '#117733'])
ax.axhline(0, color='black', linestyle='--', linewidth=0.7)
ax.set_ylabel('mean reward')
ax.set_title('Blindspot calibration + trained-policy performance')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

heldout = df_eval[df_eval['split'] == 'test'].groupby('user')['total'].agg(['mean', 'std']).sort_values('mean')
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(len(heldout)), heldout['mean'], yerr=heldout['std'], capsize=4,
       color=['#cc3333' if x < 0 else '#22aa66' for x in heldout['mean']])
ax.axhline(0, color='black', linestyle='--', linewidth=0.7)
ax.set_xticks(range(len(heldout)))
ax.set_xticklabels([str(uid)[:6] for uid in heldout.index])
ax.set_ylabel('mean reward')
ax.set_title('Held-out users only')
plt.tight_layout()
plt.show()

## 6. Hub push, cache precompute, reward ablation, and Space deploy

The final section makes the demo robust for the full judging window:

1. save the trained adapter
2. upload the Hub model card
3. precompute both before/after demo caches
4. run the reward-shaping ablation over the cached trained policy
5. deploy the CPU-only Hugging Face Space

In [ ]:
model.save_pretrained('blindspot-env/training/checkpoints/grpo')
tokenizer.save_pretrained('blindspot-env/training/checkpoints/grpo')
print('✓ saved adapter and tokenizer')

In [ ]:
import os

try:
    from google.colab import userdata
    for key in ['HF_TOKEN', 'GITHUB_TOKEN', 'OPENAI_API_KEY']:
        try:
            value = userdata.get(key)
            if value:
                os.environ[key] = value
        except Exception:
            pass
except Exception:
    pass

print({key: bool(os.environ.get(key)) for key in ['HF_TOKEN', 'GITHUB_TOKEN', 'OPENAI_API_KEY']})

In [ ]:
%%bash
cd blindspot-env
python scripts/push_to_hub.py
python scripts/precompute_demo_cache.py
python scripts/reward_ablation.py || true
python scripts/deploy_to_space.py

In [ ]:
import json
from pathlib import Path
import pandas as pd

ablation_path = Path('blindspot-env/data/reward_ablation.json')
if ablation_path.exists():
    ablation = json.load(open(ablation_path))
    rows = [
        {'variant': name, 'mean_reward': round(vals['mean'], 3), 'std': round(vals['std'], 3), 'n': vals['n']}
        for name, vals in ablation.items()
    ]
    display(pd.DataFrame(rows))
else:
    print('reward_ablation.json not found; skip display')

In [ ]:
import os
import subprocess

token = os.environ.get('GITHUB_TOKEN', '')
if token:
    os.chdir('blindspot-env')
    subprocess.run(['git', 'config', 'user.email', 'colab@blindspot.ai'], check=True)
    subprocess.run(['git', 'config', 'user.name', 'Blindspot Colab'], check=True)
    subprocess.run([
        'git', 'add',
        'data/demo_cache.json',
        'data/demo_cache_pretrain.json',
        'data/reward_ablation.json',
        'training/MODEL_CARD.md',
        'notebooks/02_training.ipynb',
        'notebooks/03_demo.ipynb',
    ], check=False)
    result = subprocess.run(['git', 'commit', '-m', 'Phase E: notebook, cache, and model-card refresh'])
    if result.returncode == 0:
        remote = f'https://{token}@github.com/vasarlalikhilavinash/blindspot-env.git'
        subprocess.run(['git', 'push', remote, 'HEAD:main'], check=True)
        print('✓ pushed updated artifacts to GitHub')
    else:
        print('(nothing new to commit)')
else:
    print('GITHUB_TOKEN missing; skip push-back step')